# **Demo 01: Building a CrewAI-Powered Agent With OpenAI**


**Objective:** To demonstrate how to build a multi-agent system using CrewAI that intelligently identifies queries requiring real-time information and retrieves the most up-to-date data accordingly.

**Prerequisites:** OpenAI key

**Tools required:** Python

**Scenario:** A product manager at a digital knowledge platform needs to manage an increasing volume of user queries, many of which require real-time updates and dynamic information. At present, these queries are reviewed and answered manually, leading to delays, inconsistencies, and rising operational costs. The goal is to automate the query-handling process using a multi-agent system that can identify when real-time data retrieval is needed and provide users with fast, accurate, and context-aware responses while maintaining efficiency and cost control. 

In [ ]:
#Step 1: Install the required libraries
!pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic
!pip install litellm



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install "crewai[azure-ai-inference]"

In [ ]:
import os
import requests
from crewai.llm import LLM
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool

In [ ]:
# Credentials are loaded from a local .env file that is never committed.
# Copy .env.example (repo root) to .env and fill in your own keys.
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# Tavily API Key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# ---- Custom CrewAI Tool for Web Search ----
class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload)
        data = response.json()

        results = []
        for r in data["results"]:
            results.append(f"{r['title']} - {r['url']}")

        return "\n".join(results)


search_tool = TavilySearchTool()

# ---- Azure LLM ----
llm = LLM(
    model="azure/gpt-5-mini",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version="2024-12-01-preview",
    is_litellm=True,
    temperature=0,
    max_completion_tokens=3500
)

In [ ]:
# Researcher agent
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for FMCG",
    backstory="You are an expert in artificial intelligence and stay updated with the latest research trends in FMCG.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=1,
    tools=[search_tool]
)

# Writer agent
writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory="You are an experienced technical writer with expertise in summarizing research for executives.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    tools=[search_tool]
)


In [ ]:
# ---- Tasks ----
task_research = Task(
    description="Search the web and identify the top 3 recent advancements in AI for FMCG.",
    expected_output="Detailed notes explaining three recent AI advancements in FMCG with examples.",
    agent=researcher
)

task_write = Task(
    description="""
Write a concise executive summary using the research notes.

Requirements:
- Maximum 100 words
- Use bullet points
- Focus only on the 3 key advancements
""",
    expected_output="Executive summary of AI advancements in FMCG.",
    agent=writer,
    context=[task_research]
)

In [ ]:
# ---- Crew ----
crew = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

result = await crew.kickoff_async()

print("\nFinal Output:\n")
print(result.raw)